In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from rdkit import Chem
import faiss
import torch


In [3]:
ROOT = Path("..")
DATA = ROOT / "data"
EMB = ROOT / "embeddings"

PUBCHEM_CLEAN = DATA / "pubchem_clean.csv"
MOLCLS_CLEAN = DATA / "molecule_classification_clean.csv"
DRUGBANK_PARSED = DATA / "drugbank_parsed.csv"

print("Using cleaned files:")
print(PUBCHEM_CLEAN)
print(MOLCLS_CLEAN)
print(DRUGBANK_PARSED)

pubchem_df = pd.read_csv(PUBCHEM_CLEAN)
mcls_df = pd.read_csv(MOLCLS_CLEAN)
drugbank_df = pd.read_csv(DRUGBANK_PARSED)


Using cleaned files:
..\data\pubchem_clean.csv
..\data\molecule_classification_clean.csv
..\data\drugbank_parsed.csv


In [4]:
pubchem_smiles = pubchem_df["SMILES"].dropna().astype(str).tolist()
print("Valid PubChem molecules:", len(pubchem_smiles))


Valid PubChem molecules: 196185


In [5]:
from transformers import AutoTokenizer, AutoModel

MODEL = "seyonec/PubChem10M_SMILES_BPE_450k"

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL).to(device)
model.eval()


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(52000, 768, padding_idx=1)
    (position_embeddings): Embedding(512, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-5): 6 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (drop

In [13]:
def embed_smiles_batch(smiles, batch=32):
    max_len = 512  # model limit
    vectors = []

    with torch.no_grad():
        for i in tqdm(range(0, len(smiles), batch), desc="Embedding SMILES"):
            chunk = smiles[i:i+batch]

            # Remove None / empty
            chunk = [s for s in chunk if isinstance(s, str) and len(s) > 0]
            if len(chunk) == 0:
                continue

            try:
                enc = tokenizer(
                    chunk,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=max_len
                ).to(device)

                out = model(**enc)

                emb = out.last_hidden_state.mean(1).detach().cpu().numpy()
                vectors.append(emb)

                # cleanup safely
                del enc, out
                torch.cuda.empty_cache()

            except RuntimeError as e:
                print("⚠️ RuntimeError:", e)

                # If the batch is too big, retry with smaller batch size
                if batch > 1:
                    print("🔁 Retrying with smaller batch:", batch // 2)
                    return embed_smiles_batch(smiles, batch=batch // 2)
                else:
                    print("❌ Skipping SMILES because it cannot be embedded:", chunk)
                    continue

    return np.vstack(vectors)


In [14]:
pubchem_smiles = [s for s in pubchem_smiles if isinstance(s, str) and len(s) < 300]
print("Filtered valid SMILES count:", len(pubchem_smiles))


Filtered valid SMILES count: 192783


In [15]:
embeddings = embed_smiles_batch(pubchem_smiles, batch=32)
np.save(EMB / "pubchem_embeddings.npy", embeddings.astype("float32"))

print("Embeddings shape:", embeddings.shape)


Embedding SMILES:   0%|          | 0/6025 [00:00<?, ?it/s]

Embeddings shape: (192783, 768)


In [24]:
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings.astype("float32"))

faiss.write_index(index, str(EMB/"pubchem_index.faiss"))

pubchem_df.to_parquet(EMB/"pubchem_metadata.parquet", index=False)

print("Saved pubchem index + metadata")


Saved pubchem index + metadata


In [25]:
idx = faiss.read_index(str(EMB/"pubchem_index.faiss"))
meta = pd.read_parquet(EMB/"pubchem_metadata.parquet")

def search_smiles(q,k=5):
    vec = embed_smiles_batch([q])
    D,I = idx.search(vec.astype("float32"), k)
    return meta.iloc[I[0]].assign(score=D[0])


In [26]:
from sentence_transformers import SentenceTransformer
text_model = SentenceTransformer("all-MiniLM-L6-v2")

descs = drugbank_df["description"].fillna("").astype(str).tolist()
text_vecs = text_model.encode(descs, convert_to_numpy=True, show_progress_bar=True)

faiss.normalize_L2(text_vecs)

text_idx = faiss.IndexFlatIP(text_vecs.shape[1])
text_idx.add(text_vecs.astype("float32"))

faiss.write_index(text_idx, str(EMB/"drugbank_text_index.faiss"))
drugbank_df.to_parquet(EMB/"drugbank_text_metadata.parquet")

print("Saved DrugBank text index + metadata")


Batches:   0%|          | 0/2039 [00:00<?, ?it/s]

Saved DrugBank text index + metadata


In [27]:
def text_query(q,k=5):
    qvec = text_model.encode([q],convert_to_numpy=True)
    faiss.normalize_L2(qvec)
    D,I = text_idx.search(qvec.astype("float32"),k)
    return drugbank_df.iloc[I[0]][["name","description"]].assign(score=D[0])

text_query("EGFR inhibitor",k=5)


,name,description,score
58477,Pelitinib,"Pelitinib (EKB-569) is a potent, low molecular...",0.706807
62620,Osimertinib,"Osimertinib is an oral, third-generation epide...",0.645165
64178,Zalutumumab,Zalutumumab (proposed trade name HuMax-EGFR) i...,0.597005
58066,INCB7839,"INCB7839 is a novel, orally available ADAM met...",0.592720
63720,Icotinib,Icotinib is a potent and specific epidermal gr...,0.587963


In [29]:
# import sys, torch
# print("sys.executable:", sys.executable)
# print("torch:", torch.__version__, "cuda_available:", torch.cuda.is_available())


In [30]:
# # Cell 1
# import os
# from pathlib import Path
# import pandas as pd
# import numpy as np
# from tqdm.auto import tqdm
# from rdkit import Chem
# from rdkit.Chem import Descriptors, QED
# import faiss
# import torch
# from transformers import AutoTokenizer, AutoModel
# from bs4 import BeautifulSoup

In [31]:


# # Paths relative to notebook location (notebook in notebooks/)
# ROOT = Path("..")  # project root
# DATA_DIR = ROOT / "data"
# EMB_DIR = ROOT / "embeddings"
# NOTEBOOKS_DIR = ROOT / "notebooks"

# EMB_DIR.mkdir(parents=True, exist_ok=True)

# PUBCHEM_PATH = DATA_DIR / "pubchem.csv"
# DRUGBANK_PATH = DATA_DIR / "drugbank.xml"
# MOLEC_CLS_PATH = DATA_DIR / "molecule_classification_dataset.csv"

# print("Using paths:")
# print("PUBCHEM:", PUBCHEM_PATH.resolve())
# print("DRUGBANK:", DRUGBANK_PATH.resolve())
# print("MOLEC CLS:", MOLEC_CLS_PATH.resolve())


In [32]:
# # Cell 2
# def safe_read_csv(path):
#     print(f"Loading {path} ...")
#     df = pd.read_csv(path, low_memory=False)
#     print("Shape:", df.shape)
#     return df

# pubchem_df = safe_read_csv(PUBCHEM_PATH)
# molec_cls_df = safe_read_csv(MOLEC_CLS_PATH)

# def detect_smiles_col(df):
#     candidates = ["smiles","SMILES","canonical_smiles","isomeric_smiles","SMILES_STRING","canonical_smiles"]
#     for c in df.columns:
#         if c.lower() in [x.lower() for x in candidates]:
#             return c
#     # heuristic: column with SMILES-like characters
#     for c in df.columns:
#         sample = df[c].dropna().astype(str)
#         if len(sample) > 0:
#             s = sample.iloc[0]
#             if any(ch in s for ch in ["(",")","=","#","/","\\","@","[","]"]):
#                 return c
#     return None

# pubchem_smiles_col = detect_smiles_col(pubchem_df)
# molec_cls_smiles_col = detect_smiles_col(molec_cls_df)

# print("Detected SMILES columns:")
# print("pubchem:", pubchem_smiles_col)
# print("molecule_classification:", molec_cls_smiles_col)


In [33]:
# # Cell 3
# def validate_and_props(smiles_series):
#     out = []
#     for s in tqdm(smiles_series.fillna("").astype(str), desc="Validating SMILES"):
#         s = s.strip()
#         valid = False
#         mw = None
#         qed = None
#         try:
#             mol = Chem.MolFromSmiles(s)
#             if mol:
#                 valid = True
#                 mw = Descriptors.MolWt(mol)
#                 qed = QED.qed(mol)
#         except Exception:
#             valid = False
#         out.append((s, valid, mw, qed))
#     return pd.DataFrame(out, columns=["SMILES","RDKit_valid","MolWeight","QED"])

# if pubchem_smiles_col is None:
#     raise ValueError("Could not detect SMILES column in pubchem.csv — open the CSV and tell me the column name.")
# pubchem_min = pubchem_df[[pubchem_smiles_col]].rename(columns={pubchem_smiles_col: "SMILES"})
# pubchem_min["SMILES"] = pubchem_min["SMILES"].astype(str).str.strip()
# pubchem_valid = validate_and_props(pubchem_min["SMILES"])
# pubchem_out = pd.concat([pubchem_min.reset_index(drop=True), pubchem_valid[["RDKit_valid","MolWeight","QED"]]], axis=1)
# print("PubChem rows:", len(pubchem_out), "valid SMILES:", pubchem_out["RDKit_valid"].sum())
# pubchem_out = pubchem_out[pubchem_out["RDKit_valid"]].drop_duplicates(subset=["SMILES"]).reset_index(drop=True)
# print("After dedupe valid count:", len(pubchem_out))

# # Molecule classification dataset (optional)
# if molec_cls_smiles_col:
#     mc_min = molec_cls_df[[molec_cls_smiles_col]].rename(columns={molec_cls_smiles_col: "SMILES"})
#     mc_min["SMILES"] = mc_min["SMILES"].astype(str).str.strip()
#     mc_valid = validate_and_props(mc_min["SMILES"])
#     molec_out = pd.concat([mc_min.reset_index(drop=True), mc_valid[["RDKit_valid","MolWeight","QED"]]], axis=1)
#     molec_out = molec_out[molec_out["RDKit_valid"]].drop_duplicates(subset=["SMILES"]).reset_index(drop=True)
#     print("Molecule classification dataset valid count:", len(molec_out))
# else:
#     molec_out = pd.DataFrame(columns=["SMILES"])
#     print("No SMILES detected in molecule_classification_dataset.csv")


In [34]:
# # Stream-parse DrugBank XML safely and write parsed rows to CSV
# # Paste into a notebook cell or run as a script.
# import xml.etree.ElementTree as ET
# from pathlib import Path
# from tqdm.auto import tqdm
# import csv
# from rdkit import Chem
# import sys

# # === Config ===
# NOTEBOOK_LOC = Path(".")  # set to "." if running from project root; if running from notebooks/, use Path("..")
# DATA_DIR = NOTEBOOK_LOC / ".." / "data" if (NOTEBOOK_LOC / ".." / "data").exists() else Path("data")
# DRUGBANK_PATH = DATA_DIR / "drugbank.xml"
# OUT_CSV = DATA_DIR / "drugbank_parsed.csv"

# BATCH_WRITE = 1000         # number of records to buffer before writing to CSV
# VALIDATE_SMILES = False    # set True to validate SMILES using RDKit during parse (slower)

# # Safety checks
# if not DRUGBANK_PATH.exists():
#     raise FileNotFoundError(f"DrugBank file not found at: {DRUGBANK_PATH.resolve()}")

# print("Parsing DrugBank from:", DRUGBANK_PATH)
# print("Output will be written to:", OUT_CSV)
# print("Batch write size:", BATCH_WRITE, "Validate SMILES:", VALIDATE_SMILES)

# # Helper functions (namespace-robust)
# def find_text(elem, tag_localname):
#     """Return text of first child whose local-name equals tag_localname (ignores namespace)."""
#     # direct children
#     for child in elem:
#         # tag may be like '{namespace}localname' or 'localname'
#         if child.tag.endswith("}" + tag_localname) or child.tag == tag_localname:
#             return child.text
#     # fallback: search deeper with findall and test localname
#     for child in elem.iter():
#         t = child.tag
#         if t.endswith("}" + tag_localname) or t == tag_localname:
#             return child.text
#     return None

# def findall_elems(elem, path_local):
#     """Find all elements by a simple local-name based path (e.g., 'calculated-properties/property')."""
#     parts = path_local.split("/")
#     # naive recursive search
#     results = []
#     def rec(current, idx):
#         if idx >= len(parts):
#             results.append(current)
#             return
#         tag = parts[idx]
#         for child in current:
#             if child.tag.endswith("}" + tag) or child.tag == tag:
#                 rec(child, idx+1)
#     rec(elem, 0)
#     return results

# # Prepare CSV: overwrite if exists
# with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
#     writer = csv.DictWriter(f, fieldnames=["drugbank_ids","name","SMILES","inchi","description","targets","RDKit_valid"])
#     writer.writeheader()

# # Stream-parse
# context = ET.iterparse(str(DRUGBANK_PATH), events=("end",))
# buffer = []
# count = 0
# # iterparse yields many events; we'll pick elements whose tag localname == 'drug'
# for event, elem in tqdm(context, desc="Streaming DrugBank XML"):
#     tag = elem.tag
#     if tag.endswith("}drug") or tag == "drug":
#         # parse this <drug> element
#         try:
#             # drugbank-ids (may be several)
#             db_ids = []
#             for id_tag in elem.findall(".//"):
#                 # iterate children and pick drugbank-id localname matches
#                 if id_tag.tag.endswith("}drugbank-id") or id_tag.tag == "drugbank-id":
#                     if id_tag.text:
#                         db_ids.append(id_tag.text.strip())
#             # name
#             name = find_text(elem, "name") or ""
#             name = name.strip() if name else ""

#             # description (may be long)
#             description = find_text(elem, "description") or ""
#             description = description.strip()

#             # calculated properties -> search for property/kind/value pairs
#             smiles = None
#             inchi = None
#             # find property elements under calculated-properties
#             # We'll inspect children of elem for any 'calculated-properties' parent first
#             calc_props = []
#             # Try local-name based approach
#             for child in elem:
#                 if child.tag.endswith("}calculated-properties") or child.tag == "calculated-properties":
#                     for prop in child:
#                         calc_props.append(prop)
#             # fallback: search deeper if not found
#             if not calc_props:
#                 for node in elem.iter():
#                     if node.tag.endswith("}calculated-properties") or node.tag == "calculated-properties":
#                         for prop in node:
#                             calc_props.append(prop)
#             # parse each property element for kind/value
#             for prop in calc_props:
#                 # find kind and value under prop
#                 kind_text = None
#                 value_text = None
#                 for sub in prop:
#                     if sub.tag.endswith("}kind") or sub.tag == "kind":
#                         kind_text = sub.text
#                     if sub.tag.endswith("}value") or sub.tag == "value":
#                         value_text = sub.text
#                 if kind_text and value_text:
#                     kk = kind_text.strip().lower()
#                     if "smiles" in kk and not smiles:
#                         smiles = value_text.strip()
#                     if "inchi" in kk and not inchi:
#                         inchi = value_text.strip()

#             # targets: collect polypeptide names or external identifiers
#             targets = []
#             # find targets parent(s)
#             for tnode in elem.iter():
#                 if tnode.tag.endswith("}targets") or tnode.tag == "targets":
#                     for t in tnode:
#                         # look for polypeptide inside target
#                         for sub in t.iter():
#                             if sub.tag.endswith("}polypeptide") or sub.tag == "polypeptide":
#                                 # prefer polypeptide name
#                                 pname = find_text(sub, "name")
#                                 if pname:
#                                     targets.append(pname.strip())
#                                 else:
#                                     # find external-identifier/identifier
#                                     for ext in sub.iter():
#                                         if ext.tag.endswith("}external-identifier") or ext.tag == "external-identifier":
#                                             ident = find_text(ext, "identifier")
#                                             if ident:
#                                                 targets.append(ident.strip())
#             # simple RDKit validation flag (optional)
#             rdkit_ok = ""
#             if VALIDATE_SMILES and smiles:
#                 try:
#                     m = Chem.MolFromSmiles(smiles)
#                     rdkit_ok = bool(m)
#                 except Exception:
#                     rdkit_ok = False

#             row = {
#                 "drugbank_ids": "|".join(db_ids) if db_ids else "",
#                 "name": name,
#                 "SMILES": smiles or "",
#                 "inchi": inchi or "",
#                 "description": description,
#                 "targets": "|".join(targets) if targets else "",
#                 "RDKit_valid": rdkit_ok
#             }
#             buffer.append(row)
#             count += 1
#         except Exception as e:
#             # log and skip problematic entry
#             print("Parse error for one entry:", e, file=sys.stderr)

#         # write buffered rows periodically
#         if len(buffer) >= BATCH_WRITE:
#             with open(OUT_CSV, "a", newline="", encoding="utf-8") as f:
#                 writer = csv.DictWriter(f, fieldnames=["drugbank_ids","name","SMILES","inchi","description","targets","RDKit_valid"])
#                 for r in buffer:
#                     writer.writerow(r)
#             buffer = []

#         # clear the processed element to free memory
#         elem.clear()

# # write any leftover buffer
# if buffer:
#     with open(OUT_CSV, "a", newline="", encoding="utf-8") as f:
#         writer = csv.DictWriter(f, fieldnames=["drugbank_ids","name","SMILES","inchi","description","targets","RDKit_valid"])
#         for r in buffer:
#             writer.writerow(r)
#     buffer = []

# print(f"Done parsing. Total drug entries processed: {count}")
# print(f"Parsed CSV saved to: {OUT_CSV.resolve()}")


In [35]:
# # Cell 5
# from transformers import AutoTokenizer, AutoModel

# SMILES_MODEL = "seyonec/PubChem10M_SMILES_BPE_450k"  # good balance for SMILES embedding
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("Device:", device)

# tokenizer = AutoTokenizer.from_pretrained(SMILES_MODEL)
# model = AutoModel.from_pretrained(SMILES_MODEL).to(device)
# model.eval()

# def embed_smiles_list(smiles_list, batch_size=64):
#     embeddings = []
#     with torch.no_grad():
#         for i in tqdm(range(0, len(smiles_list), batch_size), desc="Embedding batches"):
#             batch = smiles_list[i:i+batch_size]
#             enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
#             out = model(**enc)
#             vecs = out.last_hidden_state.mean(dim=1).cpu().numpy()
#             embeddings.append(vecs)
#     return np.vstack(embeddings)


cell 6 short not use

In [36]:
# # ==== NEW WORKING EMBEDDING CELL ====

# smiles_list = pubchem_out["SMILES"].tolist()

# # Use ONLY first 2000 molecules
# subset = smiles_list[:2000]
# print("Embedding only first:", len(subset))

# smiles_embeddings = embed_smiles_list(subset, batch_size=32)

# print("Embeddings shape:", smiles_embeddings.shape)

# # <-- FIX: Ensure directory exists
# EMB_DIR.mkdir(parents=True, exist_ok=True)

# # Save embeddings
# np.save(EMB_DIR / "pubchem_embeddings.npy", smiles_embeddings.astype("float32"))
# print("Saved pubchem_embeddings.npy")


chunk cell 6 use this

In [ ]:
# print("model max position embeddings:", model.config.max_position_embeddings)


model max position embeddings: 512


In [37]:
# # new_embed_smiles_list: robust chunked embedding + incremental save
# import os, gc, math
# import numpy as np
# import torch
# from tqdm.auto import tqdm

# def embed_smiles_list_robust(
#     smiles_list,
#     model,
#     tokenizer,
#     out_dir=EMB_DIR,
#     prefix="pubchem_embeddings_chunk",
#     batch_size=64,
#     checkpoint_every=500,   # how many SMILES (not batches) between status saves
#     device=None,
#     resume=False
# ):
#     """
#     Embed list of SMILES robustly in chunks and save chunk files as .npy.
#     - Enforces tokenizer/model max length (truncation).
#     - Saves each batch as a chunk file to avoid losing progress.
#     - Retries with smaller batch sizes on failure.
#     - If resume=True, will skip already-saved chunks.
#     Returns: list of chunk filenames (full paths)
#     """
#     device = device or ("cuda" if torch.cuda.is_available() else "cpu")
#     model.to(device)
#     model.eval()

#     # determine safe max length (prefer model.config.max_position_embeddings if present)
#     if hasattr(model, "config") and getattr(model.config, "max_position_embeddings", None):
#         max_len = model.config.max_position_embeddings
#     else:
#         max_len = tokenizer.model_max_length if tokenizer.model_max_length is not None else 512
#     # Some tokenizers report huge model_max_length; clamp reasonably
#     max_len = min(max_len, 2048)

#     os.makedirs(out_dir, exist_ok=True)
#     chunk_files = []

#     # compute number of batches
#     n = len(smiles_list)
#     batches = list(range(0, n, batch_size))

#     # if resume, detect already saved chunks and skip
#     if resume:
#         existing = set()
#         for fname in os.listdir(out_dir):
#             if fname.startswith(prefix) and fname.endswith(".npy"):
#                 existing.add(int(fname.split("_")[-1].split(".")[0]))  # expects prefix_idx.npy
#         # build list of batch start indices to skip if their chunk index exists
#     for bi, start in enumerate(tqdm(batches, desc="Embed batches (robust)")):
#         chunk_idx = bi  # chunk index (0-based by batch)
#         chunk_fname = Path(out_dir) / f"{prefix}_{chunk_idx}.npy"
#         if resume and chunk_fname.exists():
#             chunk_files.append(str(chunk_fname))
#             continue

#         end = min(start + batch_size, n)
#         batch = smiles_list[start:end]

#         # try encoding & model forward; on failure, retry smaller batch sizes
#         attempt_bs = len(batch)
#         success = False
#         while not success:
#             try:
#                 # tokenizer: ensure truncation to model length
#                 enc = tokenizer(
#                     batch,
#                     return_tensors="pt",
#                     padding=True,
#                     truncation=True,
#                     max_length=max_len,
#                 )
#                 # move to device
#                 enc = {k: v.to(device) for k, v in enc.items()}

#                 with torch.no_grad():
#                     out = model(**enc)

#                 # pooling: mean over token dim (last_hidden_state)
#                 # handle models that return tuple vs BaseModelOutput
#                 if hasattr(out, "last_hidden_state"):
#                     hid = out.last_hidden_state
#                 elif isinstance(out, tuple):
#                     hid = out[0]
#                 else:
#                     raise RuntimeError("Unrecognized model output, cannot get embeddings")

#                 vecs = hid.mean(dim=1).cpu().numpy().astype("float32")

#                 # save this batch as a chunk file
#                 np.save(chunk_fname, vecs)
#                 chunk_files.append(str(chunk_fname))

#                 # free memory
#                 del enc, out, hid, vecs
#                 torch.cuda.empty_cache()
#                 gc.collect()

#                 success = True

#             except RuntimeError as e:
#                 # if OOM or mismatch, reduce batch size
#                 err = str(e).lower()
#                 if "expanded size" in err or "position_ids" in err or "length" in err or "out of memory" in err:
#                     if attempt_bs <= 1:
#                         # As last resort, try truncating characters of SMILES strings to reduce token length
#                         # (not ideal but prevents stalls)
#                         shortened = [s[:max_len*3] for s in batch]  # crude char-level cut
#                         batch = shortened
#                         attempt_bs = 1
#                         continue
#                     else:
#                         # shrink batch into two halves and retry on the first half,
#                         # but we will process by subdividing: split the batch into two and put back tasks
#                         # into the batch loop by inserting new smaller batches into batches list
#                         half = attempt_bs // 2
#                         # replace current batch in processing sequence:
#                         # adjust future batches by inserting subdivided ranges
#                         # create two temporary smaller batches: process current half now, push remainder to next iterations
#                         # Simpler approach: set attempt_bs to half and batch = first half
#                         attempt_bs = half
#                         batch = batch[:attempt_bs]
#                         continue
#                 else:
#                     # unexpected runtime error -> re-raise after logging
#                     raise

#         # periodic info (checkpoint_every is in items processed)
#         if (end % checkpoint_every) == 0 or chunk_idx % 10 == 0:
#             print(f"Processed up to index {end}/{n} — saved {chunk_fname.name}")

#     return chunk_files

# # ---------------------------
# # Usage: (replace earlier call)
# # ---------------------------
# smiles_list = pubchem_out["SMILES"].tolist()
# print("Number of SMILES to embed:", len(smiles_list))

# # smaller batch size helps with GPU memory; set to 32 or 16 if you experience issues
# chunk_files = embed_smiles_list_robust(
#     smiles_list,
#     model=model,
#     tokenizer=tokenizer,
#     out_dir=str(EMB_DIR),
#     prefix="pubchem_embeddings_chunk",
#     batch_size=64,    # drop to 32 or 16 if OOM
#     checkpoint_every=500,
#     device=("cuda" if torch.cuda.is_available() else "cpu"),
#     resume=True       # set True to skip existing chunk files if re-running
# )

# print("Saved chunk files:", len(chunk_files))


In [38]:
# import numpy as np
# from pathlib import Path

# chunk_paths = sorted(Path(EMB_DIR).glob("pubchem_embeddings_chunk_*.npy"), key=lambda p: int(p.stem.split("_")[-1]))
# arrays = []
# for p in chunk_paths:
#     arr = np.load(p)
#     arrays.append(arr)
# embeddings = np.vstack(arrays).astype("float32")
# print("Final embeddings shape:", embeddings.shape)
# np.save(Path(EMB_DIR) / "pubchem_embeddings.npy", embeddings)
# print("Saved merged embeddings to", Path(EMB_DIR) / "pubchem_embeddings.npy")

# # Then build FAISS index (same as you had)
# import faiss
# dim = embeddings.shape[1]
# index = faiss.IndexFlatL2(dim)
# index.add(embeddings)
# faiss.write_index(index, str(EMB_DIR / "pubchem_index.faiss"))
# print("Created and saved FAISS index.")


cell 6 old full chunk crashing

In [35]:
# # Cell 6
# smiles_list = pubchem_out["SMILES"].tolist()
# print("Number of SMILES to embed:", len(smiles_list))
# # If you have many molecules, consider processing in chunks and saving intermediate files
# smiles_embeddings = embed_smiles_list(smiles_list, batch_size=64)
# print("Embeddings shape:", smiles_embeddings.shape)
# # Save embeddings array optionally
# np.save(EMB_DIR / "pubchem_embeddings.npy", smiles_embeddings.astype("float32"))
# print("Saved embeddings array to", EMB_DIR / "pubchem_embeddings.npy")


In [39]:
# # Cell 7B: Stream chunk files and add to a FAISS index (no giant array in RAM)
# import numpy as np, faiss, gc
# from pathlib import Path
# EMB_DIR = Path(EMB_DIR) if not isinstance(EMB_DIR, Path) else EMB_DIR

# chunk_paths = sorted(EMB_DIR.glob("pubchem_embeddings_chunk_*.npy"), key=lambda p: int(p.stem.split("_")[-1]))
# if len(chunk_paths) == 0:
#     raise FileNotFoundError("No chunk files found (pubchem_embeddings_chunk_*.npy). Run embed_smiles_list_robust first or merge chunks.")

# # load first chunk to get dimension
# first = np.load(chunk_paths[0])
# dim = first.shape[1]
# print("Embedding dimension:", dim)

# # create index (FlatL2)
# index = faiss.IndexFlatL2(dim)

# total = 0
# for p in chunk_paths:
#     arr = np.load(p).astype("float32")
#     index.add(arr)
#     total += arr.shape[0]
#     print(f"Added {arr.shape[0]} vectors from {p.name}  | total so far: {total}")
#     # free memory
#     del arr
#     gc.collect()

# print("FAISS ntotal:", index.ntotal)

# # write index to disk
# faiss.write_index(index, str(EMB_DIR / "pubchem_index.faiss"))
# print("Saved FAISS index to", EMB_DIR / "pubchem_index.faiss")


In [40]:
# # Save metadata (Cell 8)
# import pandas as pd
# from pathlib import Path

# # pubchem_out should be the DataFrame you validated earlier (deduped, valid SMILES)
# # IMPORTANT: ensure len(pubchem_out) == index.ntotal
# meta_path = EMB_DIR / "pubchem_metadata.parquet"
# metadata = pd.DataFrame({
#     "row_index": pubchem_out.index.values,
#     "SMILES": pubchem_out["SMILES"].values,
#     "MolWeight": pubchem_out.get("MolWeight").values if "MolWeight" in pubchem_out.columns else [None]*len(pubchem_out),
#     "QED": pubchem_out.get("QED").values if "QED" in pubchem_out.columns else [None]*len(pubchem_out)
# })
# print("Metadata rows:", len(metadata))
# print("Index vectors:", index.ntotal)
# if len(metadata) != index.ntotal:
#     print("WARNING: metadata row count DOES NOT match number of vectors in FAISS index.")
#     print(" - Check that embedding order matches pubchem_out order and that you didn't skip or drop rows.")
# metadata.to_parquet(meta_path, index=False)
# print("Saved metadata to", meta_path)


old cell 7

In [41]:
# # Cell 7
# dim = smiles_embeddings.shape[1]
# print("Embedding dimension:", dim)

# # choose index type: IndexFlatL2 (simple) — good for prototyping
# index = faiss.IndexFlatL2(dim)
# index.add(smiles_embeddings.astype("float32"))
# print("FAISS ntotal:", index.ntotal)

# # write index to disk
# faiss.write_index(index, str(EMB_DIR / "pubchem_index.faiss"))
# print("Saved FAISS index to", EMB_DIR / "pubchem_index.faiss")

# # Save metadata: map vector position -> original DataFrame row
# metadata = pd.DataFrame({
#     "row_index": pubchem_out.index.values,
#     "SMILES": pubchem_out["SMILES"].values,
#     "MolWeight": pubchem_out["MolWeight"].values,
#     "QED": pubchem_out["QED"].values
# })
# metadata.to_parquet(EMB_DIR / "pubchem_metadata.parquet", index=False)
# print("Saved metadata to", EMB_DIR / "pubchem_metadata.parquet")


old cell 8

In [42]:
# # Cell 8
# # load index & metadata (demonstrated for fresh sessions)
# index = faiss.read_index(str(EMB_DIR / "pubchem_index.faiss"))
# metadata = pd.read_parquet(EMB_DIR / "pubchem_metadata.parquet")

# def retrieve_by_smiles(query_smiles, k=5):
#     qvec = embed_smiles_list([query_smiles]).astype("float32")
#     D, I = index.search(qvec, k)
#     ids = I[0]
#     distances = D[0]
#     rows = metadata.iloc[ids].copy().reset_index(drop=True)
#     rows["distance"] = distances
#     return rows

# # Example retrieval: choose first SMILES as self-query
# example_smiles = smiles_list[0]
# print("Query:", example_smiles)
# print(retrieve_by_smiles(example_smiles, k=5))


In [43]:
# from pathlib import Path
# import pandas as pd

# # Project root = folder where your code/data exists
# ROOT = Path(r"C:\Users\notan\OneDrive\Desktop\genai-drug-discovery-rag")

# DATA_DIR = ROOT / "data"
# DRUGBANK_PARSED = DATA_DIR / "drugbank_parsed.csv"

# print("Looking for file:", DRUGBANK_PARSED)

# if not DRUGBANK_PARSED.exists():
#     raise FileNotFoundError("drugbank_parsed.csv not found at expected path!")

# drugbank_df = pd.read_csv(DRUGBANK_PARSED)

# print("Rows:", len(drugbank_df))
# print("Columns:", drugbank_df.columns.tolist())


In [44]:
# # Cell 9 — Text embedding + FAISS index

# from sentence_transformers import SentenceTransformer
# import faiss
# import numpy as np
# import pandas as pd

# print("Embedding DrugBank descriptions...")

# # Device config
# text_model_name = "all-MiniLM-L6-v2"
# text_device = "cuda" if torch.cuda.is_available() else "cpu"

# print("Using device:", text_device)

# # Load sentence embedding model
# text_model = SentenceTransformer(text_model_name, device=text_device)

# # Extract descriptions safely
# drug_texts = drugbank_df["description"].fillna("").astype(str).tolist()
# print("Number of descriptions to embed:", len(drug_texts))

# # Generate embeddings
# text_embeddings = text_model.encode(
#     drug_texts,
#     show_progress_bar=True,
#     convert_to_numpy=True
# )

# # Normalize for cosine similarity search
# faiss.normalize_L2(text_embeddings)

# # Build FAISS index
# text_dim = text_embeddings.shape[1]
# text_index = faiss.IndexFlatIP(text_dim)  # IP on normalized = cosine similarity
# text_index.add(text_embeddings.astype("float32"))

# # Save both index + metadata
# faiss.write_index(text_index, str(EMB_DIR / "drugbank_text_index.faiss"))
# drugbank_df.to_parquet(EMB_DIR / "drugbank_text_metadata.parquet", index=False)

# print("Text embedding + FAISS index built successfully")
# print("Saved drugbank_text_index.faiss and drugbank_text_metadata.parquet")


old cell 9

In [45]:
# # Cell 9
# from sentence_transformers import SentenceTransformer

# text_model_name = "all-MiniLM-L6-v2"
# text_device = "cuda" if torch.cuda.is_available() else "cpu"
# text_model = SentenceTransformer(text_model_name, device=text_device)

# drug_texts = drugbank_df["description"].fillna("").astype(str).tolist()
# print("Embedding DrugBank descriptions:", len(drug_texts))
# text_embeddings = text_model.encode(drug_texts, show_progress_bar=True, convert_to_numpy=True)

# # normalize for cosine similarity
# faiss.normalize_L2(text_embeddings)

# text_dim = text_embeddings.shape[1]
# text_index = faiss.IndexFlatIP(text_dim)  # inner product on normalized vectors == cosine
# text_index.add(text_embeddings.astype("float32"))
# faiss.write_index(text_index, str(EMB_DIR / "drugbank_text_index.faiss"))
# drugbank_df.to_parquet(EMB_DIR / "drugbank_text_metadata.parquet", index=False)
# print("Saved text index and metadata")


In [45]:
# # Cell 10
# text_index = faiss.read_index(str(EMB_DIR / "drugbank_text_index.faiss"))
# drugbank_meta = pd.read_parquet(EMB_DIR / "drugbank_text_metadata.parquet")

# def retrieve_text_query(query, k=5):
#     qvec = text_model.encode([query], convert_to_numpy=True)
#     faiss.normalize_L2(qvec)
#     D, I = text_index.search(qvec.astype("float32"), k)
#     results = []
#     for idx, score in zip(I[0], D[0]):
#         results.append({
#             "index": int(idx),
#             "score": float(score),
#             "name": drugbank_meta.iloc[idx]["name"],
#             "description_snippet": drugbank_meta.iloc[idx]["description"][:500]
#         })
#     return pd.DataFrame(results)

# print(retrieve_text_query("EGFR inhibitor", k=5))


In [46]:
# # Cell 11 - Save retrieval system summary

# summary = {
#     "pubchem_total_raw": int(len(pubchem_df)),
#     "pubchem_valid_dedup": int(len(pubchem_out)),
#     "pubchem_index_vectors": int(index.ntotal) if 'index' in globals() else 0,
#     "drugbank_parsed": int(len(drugbank_df)),
# }

# import json

# with open(EMB_DIR / "retrieval_demo_summary.json", "w") as f:
#     json.dump(summary, f, indent=2)

# print("Saved summary:", summary)

# print("\nNEXT SUGGESTIONS:")
# print(" - integrate `embeddings/pubchem_index.faiss` with LangChain FAISS wrapper (RAG chain).")
# print(" - add simple filtering using RDKit descriptors before returning results.")
# print(" - build a Gradio UI (app/ui.py) to query by SMILES or text and display retrieved records.")
